# Assignment: Data Generation using Modelling and Simulation for Machine Learning

## Objective
1. Use a simulation tool (SimPy) to model a system.
2. Generate a synthetic dataset from the simulation.
3. Train and compare multiple ML models on the generated data.

## Methodology
- **Simulation Tool**: SimPy (Discrete-Event Simulation)
- **Scenario**: Call Center / Service Queue
- **Parameters**: Number of Agents, Arrival Interval, Service Time
- **Target**: Average Wait Time

In [ ]:
!pip install simpy pandas scikit-learn xgboost

In [ ]:
import simpy
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

## Step 1: Simulation & Data Generation

In [ ]:
def call_center(env, num_agents, service_time, arrival_interval, wait_times):
    """Process for the Call Center Simulation"""
    call_center_resource = simpy.Resource(env, capacity=num_agents)

    def customer(env, name, service_time, wait_times):
        arrival_time = env.now
        
        with call_center_resource.request() as request:
            yield request  # Wait for an agent
            
            wait_time = env.now - arrival_time
            wait_times.append(wait_time)
            
            # Service time (Randomly varied around the average)
            actual_service_time = max(1, random.expovariate(1.0 / service_time))
            yield env.timeout(actual_service_time)

    # Customer Generator
    i = 0
    while True:
        yield env.timeout(random.expovariate(1.0 / arrival_interval))
        i += 1
        env.process(customer(env, f'Customer {i}', service_time, wait_times))

def run_simulation(num_agents, arrival_interval, service_time):
    """Runs a single simulation instance and returns the average wait time"""
    env = simpy.Environment()
    wait_times = []
    # Run for 480 minutes (8 hours)
    env.process(call_center(env, num_agents, service_time, arrival_interval, wait_times))
    env.run(until=480)
    
    if len(wait_times) > 0:
        return np.mean(wait_times)
    else:
        return 0.0

In [ ]:
dataset = []
NUM_SIMULATIONS = 1000

print(f"Generating {NUM_SIMULATIONS} samples...")

for i in range(NUM_SIMULATIONS):
    # Randomly sample input parameters within realistic bounds
    num_agents = random.randint(2, 20)
    arrival_interval = random.uniform(1.0, 10.0)
    service_time = random.uniform(5.0, 30.0)
    
    avg_wait = run_simulation(num_agents, arrival_interval, service_time)
    
    dataset.append({
        'Num_Agents': num_agents,
        'Arrival_Interval': arrival_interval,
        'Service_Time': service_time,
        'Average_Wait_Time': avg_wait
    })

df = pd.DataFrame(dataset)
print("Data Generation Complete.")
df.head()

## Step 2: Machine Learning Analysis

In [ ]:
# Train-Test Split
X = df[['Num_Agents', 'Arrival_Interval', 'Service_Time']]
y = df['Average_Wait_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(objective='reg:squarederror', random_state=42),
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'SVR': SVR(),
    'KNN': KNeighborsRegressor()
}

results = []

print("Training models...")
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results.append({'Model': name, 'RMSE': rmse, 'R2 Score': r2})

results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
results_df

In [ ]:
# Visualization of Best Model
best_model_name = results_df.iloc[0]['Model']
best_model_r2 = results_df.iloc[0]['R2 Score']
print(f"Best Model: {best_model_name} (R2: {best_model_r2:.4f})")

best_model_instance = models[best_model_name]
y_pred_best = best_model_instance.predict(X_test_scaled)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred_best, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.xlabel('Actual Wait Time')
plt.ylabel(f'Predicted Wait Time ({best_model_name})')
plt.title(f'Actual vs Predicted - {best_model_name}')
plt.show()